In [ ]:
import os
if os.getcwd().split("/")[-1] != "OCMN":
    os.chdir(f"{os.getcwd()}/..")

from utils.generator import Generator, ERGenerator, BAGenerator, SFGenerator
from utils.utils import save_network, read_network, create_output_file
from matching import Matching, MultiMatching
import config
import numpy as np
import pandas as pd
import copy
from tqdm import tqdm
import networkx as nx
import time
import psutil
from scipy import linalg
import random

# Generate Synthetic Networks

In [ ]:
networks = ["base"] + [f"overlap={overlap}" for overlap in config.NETWORK_OVERLAP]

def generate_and_save_networks(k_range={"start": 2.0, "end": 10.0, "step": 0.1}):
    for k in tqdm(np.arange(k_range['start'], k_range['end'] + k_range["step"], k_range["step"])):
        for n in config.NETWORK_NODES_LIST:
            k = round(k, 2)
            
            er_generator = ERGenerator(n, k)
            er_graphs = er_generator.generate_networks(len(networks), config.NETWORK_OVERLAP)
            dir = f"ER_n={n}_k={k}"
            os.makedirs(os.path.join(config.SYNTHETIC_NET_PATH, 'ER', dir), exist_ok=True)
            for i in range(len(networks)):
                save_network(er_graphs[i], os.path.join(config.SYNTHETIC_NET_PATH, 'ER', dir, f"{networks[i]}.txt"))
            
            ba_generator = SFGenerator(n, k)
            ba_graphs = ba_generator.generate_networks(len(networks), config.NETWORK_OVERLAP)
            dir = f"BA_n={n}_k={k}"
            os.makedirs(os.path.join(config.SYNTHETIC_NET_PATH, 'BA', dir), exist_ok=True)
            for i in range(len(networks)):
                save_network(ba_graphs[i], os.path.join(config.SYNTHETIC_NET_PATH, 'BA', dir, f"{networks[i]}.txt"))

generate_and_save_networks()

In [ ]:
networks = ["base"] + [f"overlap={overlap}" for overlap in [-1]]

def generate_and_save_networks():
    for n in [100, 200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 6000, 7000, 8000, 9000, 10000]:
        k = round(4.0, 2)
        
        er_generator = ERGenerator(n, k)
        er_graphs = er_generator.generate_networks(len(networks), [-1])
        dir = f"ER_n={n}_k={k}"
        os.makedirs(os.path.join(config.SYNTHETIC_NET_PATH, 'ER', dir), exist_ok=True)
        for i in range(len(networks)):
            save_network(er_graphs[i], os.path.join(config.SYNTHETIC_NET_PATH, 'ER', dir, f"{networks[i]}.txt"))
        
        ba_generator = SFGenerator(n, k)
        ba_graphs = ba_generator.generate_networks(len(networks), [-1])
        dir = f"BA_n={n}_k={k}"
        os.makedirs(os.path.join(config.SYNTHETIC_NET_PATH, 'BA', dir), exist_ok=True)
        for i in range(len(networks)):
            save_network(ba_graphs[i], os.path.join(config.SYNTHETIC_NET_PATH, 'BA', dir, f"{networks[i]}.txt"))

generate_and_save_networks()

# 控制能量

In [ ]:
def create_stable_system(graph, n_nodes):
    """
    为给定拓扑生成稳定的动力学矩阵 A。
    1. 随机赋予权重 N(0, 1)
    2. 谱平移使其稳定 (Hurwitz)
    """
    # 1. 获取邻接矩阵并赋予随机权重
    A = nx.to_numpy_array(graph, nodelist=range(1, n_nodes + 1)) # 假设节点编号从1开始
    mask = A != 0
    weights = np.random.randn(np.sum(mask))
    A[mask] = weights
    
    # 2. 稳定性处理: A_stable = A - (max(real(eig)) + 1) * I
    # 计算特征值
    eigvals = np.linalg.eigvals(A)
    max_real = np.max(eigvals.real)
    
    # 稍微多移一点(1.0)，保证严格稳定，避免临界稳定性导致的数值问题
    A_stable = A - (max_real + 1.0) * np.eye(n_nodes)
    
    return A_stable

def calculate_log_energy(A, driver_nodes, n_nodes):
    """
    计算控制能量指标 log10(Trace(W^-1))
    使用 Infinite Horizon Controllability Gramian (Lyapunov Equation)
    """
    # 1. 构建输入矩阵 B (N x M)
    # driver_nodes 是节点列表，假设编号 1~N
    m_inputs = len(driver_nodes)
    if m_inputs == 0:
        return np.inf # 不可控，能量无穷大
        
    B = np.zeros((n_nodes, m_inputs))
    for col_idx, node_id in enumerate(driver_nodes):
        # 注意：Python索引从0开始，节点如果从1开始需要 -1
        row_idx = int(node_id) - 1 
        B[row_idx, col_idx] = 1.0
        
    # 2. 解 Lyapunov 方程: A W + W A^T + B B^T = 0
    # Scipy 定义为 A X + X A^H = Q，所以这里 Q = - B B^T
    Q = -np.dot(B, B.T)
    
    try:
        Wc = linalg.solve_continuous_lyapunov(A, Q)
    except linalg.LinAlgError:
        return np.nan # 数值解失败

    # 3. 计算能量指标
    # 指标：Trace(Inv(Wc))。
    # 为了数值稳定性，通常计算特征值的倒数和
    try:
        # Wc 是对称正定的，用 SVD 或 eigh
        eig_W = linalg.eigvalsh(Wc)
        # 过滤掉极小的数值噪声 (小于 1e-12 视为 0)
        eig_W = eig_W[eig_W > 1e-12]
        
        # 如果特征值数量小于 N，说明实际不可控（或者数值上奇异）
        # 但在结构可控性保证下，理论上是可控的。
        # 这里直接求逆的迹
        energy_metric = np.sum(1.0 / eig_W)
        return np.log10(energy_metric)
        
    except Exception:
        return np.nan

def energy_experiment(
    n_list=[100],
    k_range={"start": 2.0, "end": 6.0, "step": 1.0}, # 缩减密度范围
    result_columns=[
        "network_type", "N", "<k>", "seq", 
        "UDS_Size_CLAPS", "UDS_Size_RSU",
        "Energy_L1_CLAPS", "Energy_L1_RSU",
        "Energy_L2_CLAPS", "Energy_L2_RSU"
    ]
):
    output_file_name = create_output_file(result_columns, "energy_verification")
    
    for n in n_list:
        print(f"Processing Energy Experiment n={n}...")
        
        for network_type in ["ER", "BA"]:
            for k in tqdm(np.arange(k_range['start'], k_range['end'] + k_range["step"], k_range["step"])):
                k = round(k, 2)
                for seq in range(20): # 样本数可以稍微多一点，比如20
                    dir = f"{network_type}_n={n}_k={k}"
                    
                    # --- 1. 读取网络 ---
                    matchings = []

                    base_path = os.path.join(config.SYNTHETIC_NET_PATH, network_type, dir)
                    if not os.path.exists(base_path): continue
                    
                    files = [f for f in os.listdir(base_path) if "base" in f or "-1" in f]
                    # 确保只取两个层
                    files = files[:2] 
                    
                    graphs = [] # 保存 nx graph 用于生成 A
                    for file in files:
                        graph = read_network(os.path.join(base_path, file), n)
                        graphs.append(graph)
                        m = Matching(graph)
                        m.HK_algorithm()
                        matchings.append(m)

                    if len(matchings) < 2: continue

                    # --- 2. 运行优化算法获取驱动节点集 ---
                    
                    # CLAP-S
                    mm_claps = MultiMatching(copy.deepcopy(matchings))
                    mm_claps.CLAPS() # 运行优化
                    # 获取优化后的驱动节点并集
                    drivers_claps = set(mm_claps.matchings[0].driver_nodes).union(
                                    set(mm_claps.matchings[1].driver_nodes))
                    
                    # RSU
                    mm_rsu = MultiMatching(copy.deepcopy(matchings))
                    mm_rsu.RSU() # 运行优化
                    drivers_rsu = set(mm_rsu.matchings[0].driver_nodes).union(
                                  set(mm_rsu.matchings[1].driver_nodes))
                    
                    # --- 3. 动力学实证 ---
                    
                    # 生成随机稳定的状态矩阵 A (每层一个)
                    A1 = create_stable_system(graphs[0], n)
                    A2 = create_stable_system(graphs[1], n)
                    
                    # 计算 Layer 1 的控制能量
                    e_l1_claps = calculate_log_energy(A1, list(drivers_claps), n)
                    e_l1_rsu   = calculate_log_energy(A1, list(drivers_rsu), n)
                    
                    # 计算 Layer 2 的控制能量
                    e_l2_claps = calculate_log_energy(A2, list(drivers_claps), n)
                    e_l2_rsu   = calculate_log_energy(A2, list(drivers_rsu), n)
                    
                    # --- 4. 写入结果 ---
                    with open(output_file_name, "a", encoding="utf-8") as output_file:
                        output_file.write(",".join([
                            network_type, str(n), str(k), str(seq),
                            str(len(drivers_claps)), str(len(drivers_rsu)),
                            f"{e_l1_claps:.4f}", f"{e_l1_rsu:.4f}",
                            f"{e_l2_claps:.4f}", f"{e_l2_rsu:.4f}"
                        ]) + "\n")

energy_experiment()

# 删边鲁棒性

In [ ]:
def perturb_network(graph, drop_prob):
    """
    随机移除网络中 drop_prob 比例的边
    """
    if drop_prob <= 0:
        return graph.copy()
    
    G_p = graph.copy()
    edges = list(G_p.edges)
    num_to_remove = int(len(edges) * drop_prob)
    
    if num_to_remove > 0:
        edges_to_remove = random.sample(edges, num_to_remove)
        G_p.remove_edges_from(edges_to_remove)
        
    return G_p

def robustness_experiment(
    n_list=[1000],
    p_range=np.arange(0.0, 0.55, 0.05), # 删边比例 0% -> 50%
    result_columns=[
        "network_type", "N", "<k>", "p", "seq", 
        "UDS_Size_CLAPS", "UDS_Size_RSU",
        "Jaccard_Similarity_CLAPS", "Jaccard_Similarity_RSU",
        "Retention_Rate_CLAPS", "Retention_Rate_RSU"
    ],
):
    output_file_name = create_output_file(result_columns, "robustness_results")

    for n in n_list:
        # 这里为了演示，假设只用 k=4 的网络
        k = round(4.0, 2)
        
        for network_type in ["ER", "BA"]:
            dir = f"{network_type}_n={n}_k={k}"
            
            base_path = os.path.join(config.SYNTHETIC_NET_PATH, network_type, dir)
            if not os.path.exists(base_path): continue
            
            files = [f for f in os.listdir(base_path) if "base" in f or "-1" in f]
            # 确保只取两个层
            files = files[:2] 
            
            graphs_original = [] # 保存 nx graph 用于生成 A
            for file in files:
                graph = read_network(os.path.join(base_path, file), n)
                graphs_original.append(graph)
            
            matchings_original = []
            for g in graphs_original:
                m = Matching(g)
                m.HK_algorithm()
                matchings_original.append(m)

            # CLAP-S
            mm_claps = MultiMatching(copy.deepcopy(matchings_original))
            mm_claps.CLAPS() # 运行优化
            # 获取优化后的驱动节点并集
            u0_claps = set(mm_claps.matchings[0].driver_nodes).union(
                            set(mm_claps.matchings[1].driver_nodes))
            
            # RSU
            mm_rsu = MultiMatching(copy.deepcopy(matchings_original))
            mm_rsu.RSU() # 运行优化
            u0_rsu = set(mm_rsu.matchings[0].driver_nodes).union(
                            set(mm_rsu.matchings[1].driver_nodes))
            
            for p in tqdm(p_range):
                for seq in range(10):
                    # 2. 扰动网络 G_p
                    graphs_p = [perturb_network(g, p) for g in graphs_original]
                    
                    # 3. 计算扰动后的集合 U_p
                    matchings_p = []
                    for g in graphs_p:
                        m = Matching(g)
                        m.HK_algorithm()
                        matchings_p.append(m)

                    # CLAP-S
                    mm_claps_p = MultiMatching(copy.deepcopy(matchings_p))
                    mm_claps_p.CLAPS() # 运行优化
                    up_claps = set(mm_claps_p.matchings[0].driver_nodes).union(
                                set(mm_claps_p.matchings[1].driver_nodes))
                    
                    # RSU
                    mm_rsu_p = MultiMatching(copy.deepcopy(matchings_p))
                    mm_rsu_p.RSU() # 运行优化
                    up_rsu = set(mm_rsu_p.matchings[0].driver_nodes).union(
                                set(mm_rsu_p.matchings[1].driver_nodes))
                    
                    # 4. 计算指标
                    # 指标 1: Jaccard (相似度)
                    jac_claps = len(u0_claps & up_claps) / len(u0_claps | up_claps)
                    jac_rsu = len(u0_rsu & up_rsu) / len(u0_rsu | up_rsu)
                    
                    # 指标 2: Retention Rate (有多少老节点被留用了)
                    # Denominator 使用 |U_p| (如 Prompt 建议) 还是 |U_0|?
                    # 建议用 |U_0| 表示“原投资保留率”，或者 Jaccard 更综合
                    ret_claps = len(u0_claps & up_claps) / len(up_claps) if len(up_claps) > 0 else 0
                    ret_rsu = len(u0_rsu & up_rsu) / len(up_rsu) if len(up_rsu) > 0 else 0
                    
                    # 写入结果
                    with open(output_file_name, "a", encoding="utf-8") as output_file:
                        output_file.write(",".join([
                            network_type, str(n), str(k), str(round(p, 2)), str(seq),
                            str(len(up_claps)), str(len(up_rsu)),
                            f"{jac_claps:.4f}", f"{jac_rsu:.4f}",
                            f"{ret_claps:.4f}", f"{ret_rsu:.4f}"
                        ]) + "\n")

robustness_experiment()

# RSU 不同的采样次数的敏感性分析

In [ ]:
def rsu_k_ablation_experiment(
    network_settings=[
        {"network_type": "ER", "n": 1000, "k": 2.0},
        {"network_type": "BA", "n": 1000, "k": 2.0},
        {"network_type": "ER", "n": 1000, "k": 3.0},
        {"network_type": "BA", "n": 1000, "k": 3.0},
        {"network_type": "ER", "n": 1000, "k": 4.0},
        {"network_type": "BA", "n": 1000, "k": 4.0},
        {"network_type": "ER", "n": 1000, "k": 5.0},
        {"network_type": "BA", "n": 1000, "k": 5.0},
        {"network_type": "ER", "n": 1000, "k": 6.0},
        {"network_type": "BA", "n": 1000, "k": 6.0},
        {"network_type": "ER", "n": 1000, "k": 7.0},
        {"network_type": "BA", "n": 1000, "k": 7.0},
        {"network_type": "ER", "n": 1000, "k": 8.0},
        {"network_type": "BA", "n": 1000, "k": 8.0},
        {"network_type": "ER", "n": 1000, "k": 9.0},
        {"network_type": "BA", "n": 1000, "k": 9.0},
        {"network_type": "ER", "n": 1000, "k": 10.0},
        {"network_type": "BA", "n": 1000, "k": 10.0},
    ],
    s_list=[20, 50, 100, 200],
    result_columns=[
        "network_type", "N", "<k>", "seq", "RSU_Sampling",
        "MDS_1", "MDS_2", 
        "Diff_MDS_1", "Diff_MDS_2", "UDS_0", 
        "UDS_CLAPS", "UDS_RSU",
        "time_CLAPS", "time_RSU",
    ]
):
    output_file_name = create_output_file(result_columns, "rsu_k_ablation")
    results = []
    for setting in network_settings:
        for seq in tqdm(range(10)):
            dir = f"{setting['network_type']}_n={setting['n']}_k={round(setting['k'], 2)}"
            print(f"Processing {dir}...")
            matchings = []
            for file in os.listdir(os.path.join(config.SYNTHETIC_NET_PATH, setting['network_type'], dir)):
                if "base" in file or "-1" in file:
                    graph = read_network(os.path.join(config.SYNTHETIC_NET_PATH, setting['network_type'], dir, file), setting['n'])
                    matching = Matching(graph)
                    matching.HK_algorithm()
                    matchings.append(matching)
            multi_matching = MultiMatching(matchings)

            # CLAPS baseline
            mm_claps = copy.deepcopy(multi_matching)
            start = time.time()
            pre_diff_mds_1_size, pre_diff_mds_2_size, pre_union_size, union_size, average_depth = mm_claps.CLAPS()
            time_clap_s = time.time() - start

            for s in s_list:
                mm_rsu = copy.deepcopy(multi_matching)
                start = time.time()
                union_size_rsu = mm_rsu.RSU(K=s)
                time_rsu = time.time() - start
                row = [
                    setting['network_type'], setting['n'], setting['k'], seq, s,
                    str(len(matchings[0].driver_nodes)), str(len(matchings[1].driver_nodes)), 
                    str(pre_diff_mds_1_size), str(pre_diff_mds_2_size), str(pre_union_size), 
                    str(union_size), str(union_size_rsu), 
                    str(time_clap_s), str(time_rsu),
                ]
                results.append(row)
                with open(output_file_name, "a", encoding="utf-8") as output_file:
                    output_file.write(",".join(map(str, row)) + "\n")
    df = pd.DataFrame(results, columns=result_columns)
    return df

rsu_k_ablation_df = rsu_k_ablation_experiment()

# 不同节点数N、重复实验

In [ ]:
def memory_usage(
    k=4,
    n_list=[100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1200, 1400, 1600, 1800, 2000, 2500, 3000, 3500, 4000, 4500, 5000, 6000, 7000, 8000, 9000, 10000],
    result_columns=[
        "network_type", "N", "<k>", "seq", 
        "MDS_1", "MDS_2", 
        "Diff_MDS_1", "Diff_MDS_2", "UDS_0", 
        "UDS_CLAPS",
        "clap_average_length", 
        "time_CLAPS",
        "mem_CLAPS (bytes)",]
):
    output_file_name = create_output_file(result_columns, "memory_usage")
    results = []
    for n in n_list:
        print(f"Processing n={n}...")
        for seq in tqdm(range(10)):
            for network_type in ["ER", "BA"]:
                dir = f"{network_type}_n={n}_k={round(k, 2)}"
                matchings = []
                for file in os.listdir(os.path.join(config.SYNTHETIC_NET_PATH, network_type, dir)):
                    if "base" in file or "-1" in file:
                        graph = read_network(os.path.join(config.SYNTHETIC_NET_PATH, network_type, dir, file), n)
                        matching = Matching(graph)
                        matching.HK_algorithm()
                        matchings.append(matching)
                multi_matching = MultiMatching(matchings)

                process = psutil.Process(os.getpid())
                max_memory_usage = 0

                initial_memory = process.memory_info().rss

                max_memory_usage = max(max_memory_usage, process.memory_info().rss - initial_memory)

                # CLAPS baseline
                start = time.time()
                pre_diff_mds_1_size, pre_diff_mds_2_size, pre_union_size, union_size, average_depth = multi_matching.CLAPS()
                time_clap_s = time.time() - start

                final_memory = process.memory_info().rss
                max_memory_usage = max(max_memory_usage, final_memory - initial_memory)

                row = [
                    network_type, n, k, seq,
                    str(len(matchings[0].driver_nodes)), str(len(matchings[1].driver_nodes)), 
                    str(pre_diff_mds_1_size), str(pre_diff_mds_2_size), str(pre_union_size), 
                    str(union_size),
                    str(average_depth),
                    str(time_clap_s),
                    str(max_memory_usage)
                ]
                results.append(row)
                with open(output_file_name, "a", encoding="utf-8") as output_file:
                    output_file.write(",".join(map(str, row)) + "\n")
    df = pd.DataFrame(results, columns=result_columns)
    return df

memory_usage()

# 不同网络类型、不同平均度k、不同节点数N、重复实验

In [ ]:
def optimization_amount(
    k_range={"start": 2, "end": 10, "step": 0.1},
    n_list=config.NETWORK_NODES_LIST,
    result_columns=[
        "network_type", "N", "<k>", "seq", 
        "MDS_1", "MDS_2", 
        "Diff_MDS_1", "Diff_MDS_2", "UDS_0", 
        "UDS_CLAPS", "UDS_RSU", "UDS_CLAPG", "UDS_ILP", 
        "clap_average_length", 
        "time_CLAPS", "time_RSU", "time_CLAPG", "time_ILP"]
):
    output_file_name = create_output_file(result_columns, "optimization_amount")

    for n in n_list:
        print(f"Processing n={n}...")

        # ER + ER; BA + BA; ER + BA
        for network_type in ["ER", "BA", "ER+BA"]:
            for k in tqdm(np.arange(k_range['start'], k_range['end'] + k_range["step"], k_range["step"])):
                k = round(k, 2)
                for seq in range(10):
                    dir = f"{network_type}_n={n}_k={k}"
                    matchings = []

                    if "+" in network_type:
                        graph_1 = read_network(os.path.join(config.SYNTHETIC_NET_PATH, "ER", f"ER_n={n}_k={k}", f"base.txt"), n)
                        matching_1 = Matching(graph_1)
                        matching_1.HK_algorithm()
                        matchings.append(matching_1)
                        graph_2 = read_network(os.path.join(config.SYNTHETIC_NET_PATH, "BA", f"BA_n={n}_k={k}", f"base.txt"), n)
                        matching_2 = Matching(graph_2)
                        matching_2.HK_algorithm()
                        matchings.append(matching_2)
                    else:
                        for file in os.listdir(os.path.join(config.SYNTHETIC_NET_PATH, network_type, dir)):
                            # 包含base或-1的文件名
                            if "base" in file or "-1" in file:
                                graph = read_network(os.path.join(config.SYNTHETIC_NET_PATH, network_type, dir, file), n)
                                matching = Matching(graph)
                                matching.HK_algorithm()
                                matchings.append(matching)
                    
                    multi_matching = MultiMatching(matchings)
                    multi_matching_rsu = copy.deepcopy(multi_matching)
                    multi_matching_glde = copy.deepcopy(multi_matching)
                    multi_matching_ilp = copy.deepcopy(multi_matching)

                    start_time = time.time()
                    pre_diff_mds_1_size, pre_diff_mds_2_size, pre_union_size, union_size, average_depth = multi_matching.CLAPS()
                    end_time = time.time()
                    time_clap_s = end_time - start_time
                    
                    start_time = time.time()
                    union_size_rsu = multi_matching_rsu.RSU()
                    end_time = time.time()
                    time_rsu = end_time - start_time

                    start_time = time.time()
                    union_size_glde = multi_matching_glde.CLAPG()
                    end_time = time.time()
                    time_glde = end_time - start_time

                    start_time = time.time()
                    union_size_ilp = multi_matching_ilp.ILP_exact(budget_mode="auto")
                    end_time = time.time()
                    time_ilp = end_time - start_time
                    
                    with open(output_file_name, "a", encoding="utf-8") as output_file:
                        output_file.write(",".join([
                            f"{network_type}+{network_type}" if "+" not in network_type else network_type, str(n), str(k), str(seq), 
                            str(len(matchings[0].driver_nodes)), str(len(matchings[1].driver_nodes)), 
                            str(pre_diff_mds_1_size), str(pre_diff_mds_2_size), str(pre_union_size), 
                            str(union_size), str(union_size_rsu), str(union_size_glde), str(union_size_ilp), 
                            str(average_depth),
                            str(time_clap_s),str(time_rsu), str(time_glde), str(time_ilp)
                        ]) + "\n")

optimization_amount()

# 不同网络类型、不同平均度k、不同网络重叠程度

In [ ]:
def optimization_proportion(
    k_range={"start": 2, "end": 10, "step": 0.1},
    n_list=config.NETWORK_NODES_LIST,
    overlap_list=config.NETWORK_OVERLAP,
    result_columns=[
        "network_type", "N", "<k>", "overlap", 
        "MDS_1", "MDS_2", 
        "Diff_MDS_1", "Diff_MDS_2", "UDS_0", 
        "UDS_CLAPS", "UDS_RSU", "UDS_CLAPG", "UDS_ILP", 
        "clap_average_length", 
        "time_CLAPS", "time_RSU", "time_CLAPG", "time_ILP"]
):
    output_file_name = create_output_file(result_columns, "optimization_proportion")
    
    for n in n_list:
        print(f"Processing n={n}...")
        for network_type in ["ER", "BA"]:
            print(f"\tProcessing {network_type}...")
            for overlap in overlap_list:
                overlap = round(overlap, 3)
                if overlap < 0:
                    continue
                for k in tqdm(np.arange(k_range['start'], k_range['end'] + k_range["step"], k_range["step"]), desc=f"\t\tOverlap={overlap}"):
                    k = round(k, 2)
                    dir = f"{network_type}_n={n}_k={k}"

                    matchings = []
                    for file in os.listdir(os.path.join(config.SYNTHETIC_NET_PATH, network_type, dir)):
                        if "base" in file or f"overlap={overlap}.txt" in file:
                            graph = read_network(os.path.join(config.SYNTHETIC_NET_PATH, network_type, dir, file), n)
                            matching = Matching(graph)
                            matching.HK_algorithm()
                            matchings.append(matching)
                    
                    multi_matching = MultiMatching(matchings)
                    multi_matching_rsuu = copy.deepcopy(multi_matching)
                    multi_matching_glde = copy.deepcopy(multi_matching)
                    multi_matching_ilp = copy.deepcopy(multi_matching)

                    start_time = time.time()
                    pre_diff_mds_1_size, pre_diff_mds_2_size, pre_union_size, union_size, average_depth = multi_matching.CLAPS()
                    end_time = time.time()
                    time_clap_s = end_time - start_time
                    
                    start_time = time.time()
                    union_size_rsuu = multi_matching_rsuu.RSU()
                    end_time = time.time()
                    time_rsuu = end_time - start_time

                    start_time = time.time()
                    union_size_glde = multi_matching_glde.CLAPG()
                    end_time = time.time()
                    time_glde = end_time - start_time

                    start_time = time.time()
                    union_size_ilp = multi_matching_ilp.ILP_exact(budget_mode="auto")
                    end_time = time.time()
                    time_ilp = end_time - start_time

                    with open(output_file_name, "a", encoding="utf-8") as output_file:
                        output_file.write(",".join([
                            f"{network_type}+{network_type}", str(n), str(k), str(overlap),
                            str(len(matchings[0].driver_nodes)), str(len(matchings[1].driver_nodes)), 
                            str(pre_diff_mds_1_size), str(pre_diff_mds_2_size), str(pre_union_size), 
                            str(union_size), str(union_size_rsuu), str(union_size_glde), str(union_size_ilp), 
                            str(average_depth),
                            str(time_clap_s),str(time_rsuu), str(time_glde), str(time_ilp)
                        ]) + "\n")

optimization_proportion()